### Setup

In [1]:
import matplotlib.pyplot as plt
import scipy as sc
from scipy.stats import pearsonr
import numpy as np
import pandas as pd
import seaborn as sns
import sys
import os
import torch
from tqdm import tqdm

In [2]:
sys.path.insert(0, os.getcwd())

from model import GPTConfig, GPT
from lib.utils import get_batch

In [3]:
device='cpu'
dataset='shakespeare_char'
checkpoint_dir='out-shakespeare-char'

### Identifiability

In [4]:
%%capture
torch.manual_seed(1337)

# load the checkpointed model state from last train save
ckpt_path = os.path.join(checkpoint_dir, 'ckpt.pt')
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
model = GPT(gptconf)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model.load_state_dict(state_dict)

model.eval() # disables dropout
model.to(device)

In [5]:
model.config

GPTConfig(block_size=32, vocab_size=65, n_layer=2, n_head=3, head_size=20, batch_size=16, n_embd=64, dropout=0.2, bias=False, mlp_width=256)

**No LRB:**

GPTConfig(block_size=32, vocab_size=65, n_layer=2, n_head=3, head_size=64, batch_size=16, n_embd=64, dropout=0.2, bias=False, mlp_width=256)

**LRB:**



In [6]:
n = model.config.block_size
h = model.config.n_head
hs = model.config.head_size

In [7]:
X, Y = get_batch('eval', os.path.join('data', dataset), device, n, model.config.batch_size)
A,T,v = model.get_matricies(X,0)

In [26]:
A_tensor = torch.as_tensor(A)
T_tensor = torch.as_tensor(T)

U, S, Vh = torch.linalg.svd(T_tensor, full_matrices=True)
r = torch.sum(S > 1e-5).item()

U_null = U[:, r:]

A_proj_null = U_null @ U_null.T @ A_tensor
A_perp = A_tensor - A_proj_null
print(A_perp)

tensor([[ 4.3475e-04,  4.3475e-04,  1.1642e-03,  ...,  1.2867e-04,
         -2.1176e-04, -3.0494e-04],
        [ 4.9884e-01,  4.9884e-01,  3.1495e-03,  ..., -3.7599e-04,
         -3.3575e-04, -3.5730e-04],
        [ 3.3309e-01,  3.3309e-01,  3.2421e-01,  ...,  7.0120e-04,
          8.0964e-04,  1.5941e-03],
        ...,
        [ 2.9078e-02,  2.9078e-02,  2.9544e-02,  ...,  1.5512e-02,
          3.2664e-03, -1.2739e-03],
        [ 3.0180e-02,  3.0180e-02,  2.9782e-02,  ...,  3.4339e-02,
          2.9631e-02,  6.6336e-03],
        [ 3.4461e-02,  3.4461e-02,  4.0120e-02,  ...,  2.9802e-02,
          3.1139e-02,  2.4463e-02]])


In [8]:
T.shape

(32, 20)

In [9]:
np.linalg.matrix_rank(T)

20

In [10]:
T_aug = np.concatenate([T, np.ones((n, 1))], axis=1)
basis_LN = sc.linalg.null_space(T_aug.T)

In [11]:
basis_LN.shape

(32, 11)

In [12]:
d = basis_LN[:,0]
a = A[:,1]

In [13]:
a.shape

(32,)

In [14]:
def get_lam_constraints(d):
    lam_max_list = []
    lam_min_list = []
    for idx in range(len(d)):
        if d[idx] < 0:
            lam_max_list.append(-a[idx] / d[idx])
        elif d[idx] > 0:
            lam_min_list.append(-a[idx] / d[idx])
    
    lam_max = np.min(lam_max_list)
    lam_min = np.max(lam_min_list)
    return lam_min, lam_max

In [15]:
for i in range(basis_LN.shape[1]):
    d = basis_LN[:,i]
    lam_min, lam_max = get_lam_constraints(d)
    
    for lam in np.linspace(lam_min, lam_max, 1000):
        a_prime = a + lam * d
        A[:,0] = a_prime
        
        vals = np.zeros(basis_LN.shape[1])
        for j in range(basis_LN.shape[1]):
            vals[j]= basis_LN[:,j].T  @ A[:,0] / basis_LN[:,j].T @ np.ones(n)

        print(np.linalg.norm(np.diff(vals)))

28.291403034216913
28.279844113607684
28.26829335341485
28.256747941312057
28.245213866811692
28.23369011970176
28.22216624649062
28.210658007862563
28.19915484927008
28.18766081338806
28.176174268421434
28.164695380330695
28.153230621091723
28.141756670694967
28.13031345951671
28.118861373905837
28.107429805051225
28.09599712743782
28.084575980351257
28.073153900497594
28.06176205302588
28.050366241092338
28.038970592779215
28.027602734685743
28.016226484813334
28.004864397046134
27.993507416420755
27.982160477376073
27.970824770700595
27.95949412648512
27.948175175121857
27.93685394809417
27.925556756061802
27.914265847454253
27.902971276192748
27.891697842407613
27.880426088802153
27.869166437669467
27.857914115024123
27.846664878839338
27.83542955469547
27.824195088167926
27.81297478632403
27.801761804408727
27.79056495840195
27.779361191118046
27.768188073899864
27.757010294180578
27.745833806931973
27.73467241726788
27.723526113237813
27.712391094385033
27.70124881739091
27.69012

22.474220863064954
22.473905542680903
22.47361233109427
22.47332370559702
22.47306542289272
22.472802381881515
22.472575988889265
22.47235060073537
22.472153941820405
22.471958331278007
22.471790395847215
22.471635003674596
22.471496581477137
22.47136994616447
22.471262899618814
22.471174456362277
22.471095797231868
22.47104066377832
22.47099175333755
22.470978758808513
22.470960929445983
22.47096808311353
22.470989854285744
22.471034674258743
22.471077523807413
22.47115239733825
22.471233747226915
22.471345992723307
22.47146110186622
22.471597399984407
22.471752768451758
22.47190851860653
22.472102255258477
22.472294996253233
22.472517560843464
22.472745046113122
22.472996235891557
22.473255591774716
22.473535116091707
22.473828526605764
22.474141703986835
22.47447620533937
22.474817752092804
22.475178998658354
22.475551865069995
22.475950979716625
22.476355914504563
22.476780780324926
22.47722065698315
22.477683157299975
22.4781542819468
22.47864852252544
22.479144931823544
22.479671

In [16]:
basis_LN[:,0].T  @ A[:,0] / basis_LN[:,0].T @ np.ones(n)

-1.3947705367162202

In [17]:
check_val = basis_LN[:,0].T  @ A[:,0] / basis_LN[:,0].T @ np.ones(n)
vals_prime = np.zeros(basis_LN.shape[1])

for j in range(basis_LN.shape[1]):
    vals_prime[j]= basis_LN[:,j].T  @ A[:,20] / basis_LN[:,j].T @ np.ones(n)
vals_prime

array([ -1.93453014,  -4.20249871,  15.33395233,  42.76741346,
        -8.29960056,  -1.44760022, -12.01374459,   5.15889251,
        -1.726527  ,  11.49000327,  -3.34955436])

In [18]:
np.linalg.norm(np.diff(vals_prime))

68.13987259595373

In [19]:
np.linalg.norm(np.diff(vals))

58.63319368656507